# Create MERlin scripts

Generates the per-experiment MERlin config/run files into `SAMPLE_DIR/merlin/`. Codebook and microscope-parameters files are NOT copied here: they're shared reference data shipped in `MERci/data/configs/merlin/{codebooks,microscope}/`, and since this `MERci/` clone already lives inside `SAMPLE_DIR/`, the slurm script below references them by their path inside this clone directly — self-contained, no separate cluster-side copy step.

Run this after notebook 06 (needs `experiment_info.yaml`).

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd

MERCI_DIR    = Path(os.getcwd()).parent.parent.parent.parent   # MERci/ (notebook lives in MERci/notebooks/before_imaging/<variant>/<acquisition>/)
SAMPLE_DIR   = MERCI_DIR.parent                  # experiment root, e.g. LT048_sample_26/
METADATA_DIR = SAMPLE_DIR / "metadata"
POSITIONS_DIR = SAMPLE_DIR / "positions"
MERLIN_DIR   = SAMPLE_DIR / "merlin"             # new per-experiment MERlin folder
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.experiment_info import load_experiment_info, positions_file_tag
from MERci.acquisition.display        import display_file
from MERci.acquisition.merlin_config import (
    resolve_codebook_filename, resolve_microscope_parameters_filename,
    build_merlin_analysis_parameters,
    create_cluster_resource_allocation, create_snakemake_parameters,
    resolve_cluster_sample_dir, short_experiment_name, create_slurm_submit_script,
)
from MERci.acquisition.pipeline_config import load_pipeline_config

# PIPELINE_ID names this variant's own pipeline.yaml -- see notebook 01's own
# comment on PIPELINE_ID/PIPELINE_CONFIG for the rationale. MICROSCOPE/
# OBJECTIVE/LIB_NAME/N_OPT below all come from experiment_info.yaml instead
# (notebook 06 already threaded them through from this same pipeline.yaml),
# so PIPELINE_CONFIG here is only needed for the analysis-task menu.
PIPELINE_ID     = "tumor_disk"
PIPELINE_CONFIG = load_pipeline_config(MERCI_DIR / "data" / "pipelines" / PIPELINE_ID / "pipeline.yaml")

info = load_experiment_info(METADATA_DIR / "experiment_info.yaml")
MICROSCOPE  = info.microscope
OBJECTIVE   = info.extra.get("objective")   # None (older experiment_info.yaml) -> resolve_microscope_parameters_filename falls back to the microscope's default

# SAMPLE_NAME is the TRUE top-level experiment id (from experiment_info.yaml,
# notebook 06) -- the same value notebooks 01-05 used to name
# positions_*.txt / data_organization_*.csv / etc., so this notebook's file
# lookups (and its own generated merlin/ filenames) stay consistent with
# what's already on disk.
SAMPLE_NAME = info.sample_name

# SHORT_NAME: SLURM-job-name-safe short form of SAMPLE_NAME (see
# short_experiment_name) -- used below to name this notebook's own
# generated merlin/ filenames instead of the full SAMPLE_NAME, so e.g.
# "merlin_slurm_BC555_sample_05.sh" becomes "merlin_slurm_s5-BC555e.sh".
SHORT_NAME = short_experiment_name(SAMPLE_NAME, info.extra.get("imaging_dir", ""))

# Token positions_path uses below -- must match notebook 02's writer, which
# names positions files positions_{POSITIONS_TAG}.txt (POSITIONS_TAG =
# SAMPLE_NAME + "_" + imaging_dir when split into sibling acquisition
# folders, else just SAMPLE_NAME), so sibling split-layout acquisitions of
# the same sample (e.g. tumor/epi vs tumor/disk) never collide on filename.
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, info.extra.get("imaging_dir", ""))

print(f"SAMPLE_NAME : {SAMPLE_NAME}")
print(f"SHORT_NAME  : {SHORT_NAME}")
print(f"MICROSCOPE  : {MICROSCOPE}")
print(f"OBJECTIVE   : {OBJECTIVE}")
print(f"lib_name    : {info.lib_name}")
print(f"data_home   : {info.data_home}")
print(f"merlin_home : {info.merlin_home}")
print(f"folder_name : {info.folder_name}")

## Resolve shared reference files (codebook, microscope parameters)

These live in `MERci/data/configs/merlin/` — shipped with the repo, not regenerated per experiment. As a sanity check, the codebook's own `bit_names` row is compared against this experiment's bit count from `round_bit_color_map.csv`, to catch a mismatched codebook before it reaches the cluster.

In [ ]:
from MERci.acquisition.merlin_config import load_sequential_gene_names, resolve_sequential_codebook_filename

codebook_path   = MERCI_DIR / "data" / "configs" / "merlin" / "codebooks" / resolve_codebook_filename(info.lib_name)
microscope_path = MERCI_DIR / "data" / "configs" / "merlin" / "microscope" / resolve_microscope_parameters_filename(MICROSCOPE, OBJECTIVE)

for p in (codebook_path, microscope_path):
    if not p.exists():
        raise FileNotFoundError(
            f"{p} not found -- add it to MERci/data/configs/merlin/ before running this notebook."
        )

rbc_path = METADATA_DIR / "round_bit_color_map.csv"
sequential_genes_path = METADATA_DIR / "sequential_genes.csv"   # per-experiment override, optional
SEQUENTIAL_KIND = PIPELINE_CONFIG.merlin.sequential_kind
sequential_gene_names = load_sequential_gene_names(codebook_path.parent, info.lib_name, sequential_genes_path, kind=SEQUENTIAL_KIND)

# This pipeline's MERlin recipe has no barcode-decode task (see pipeline.yaml's
# merlin.analysis.tasks -- this pipeline never images a combinatorial
# barcode), so the codebook's own bit_names count has nothing to reconcile
# against round_bit_color -- lib_name is still used for the codebook/
# microscope-parameters file paths above and for this pipeline's own
# {SEQUENTIAL_KIND} companion (target gene names), just not for a
# barcode-count check.
DOES_DECODE = PIPELINE_CONFIG.merlin.tasks.get("decode", False)
n_sequential_bits = len(sequential_gene_names)
if not DOES_DECODE:
    n_bits = int(pd.read_csv(rbc_path)["bit"].nunique()) if rbc_path.exists() else None
    print(f"No barcode-decode task in this pipeline's recipe -- skipping the codebook "
          f"bit-count check. {n_sequential_bits} {SEQUENTIAL_KIND} target(s) from "
          f"{resolve_sequential_codebook_filename(info.lib_name, SEQUENTIAL_KIND)}.")
elif rbc_path.exists():
    # Distinct bit count (NOT round.max() -- a round can carry several bits,
    # one per color, so round.max() is the hyb-round count, not the bit count).
    n_bits = int(pd.read_csv(rbc_path)["bit"].nunique())
    n_barcode_bits = n_bits - n_sequential_bits
    codebook_bit_names_line = next(
        line for line in codebook_path.read_text().splitlines() if line.startswith("bit_names")
    )
    codebook_n_bits = len(codebook_bit_names_line.split(",")) - 1
    if codebook_n_bits != n_barcode_bits:
        print(f"WARNING: codebook {codebook_path.name} has {codebook_n_bits} bits, "
              f"but this experiment images {n_barcode_bits} barcode bits (of {n_bits} total) "
              f"-- double-check lib_name/codebook choice.")
    else:
        print(f"Bit count OK: {n_barcode_bits} barcode bits (of {n_bits} total imaged, "
              f"{n_sequential_bits} {SEQUENTIAL_KIND}), matching {codebook_path.name}"
              + (f" + {resolve_sequential_codebook_filename(info.lib_name, SEQUENTIAL_KIND)}" if n_sequential_bits else "") + ".")
else:
    n_bits = None
    print(f"WARNING: {rbc_path} not found -- run notebook 03 first.")

print(f"Codebook  : {codebook_path}")
print(f"Microscope: {microscope_path}")

## MERlin analysis-parameters JSON

Built from an explicit recipe (`data/configs/merlin/analysis/recipes/`) of atomic MERlin tasks (`data/configs/merlin/analysis/tasks/`) instead of copying and hand-editing a prior experiment's file -- see `build_merlin_analysis_parameters`'s docstring for the recipe/atom format, and each `.yaml` file directly to see/tune its parameters.

In [ ]:
# Unlike tumor_epi, this pipeline's smfish_signal/sum_signal are NOT derived
# per experiment -- pipeline.yaml's merlin.analysis.tasks/overrides already
# fix them (CENPA via smFISH, gamma-H2AX via SumSignal; sequential_gene_names
# above is just this pipeline's own target list, read for display/bit-count
# purposes). The pipeline's own task menu is the source of truth for which
# atoms run -- written out below as this experiment's own recipe file
# instead of the old hardcoded "default_with_segmentation.yaml" + extra_tasks
# list, so it always matches pipeline.yaml exactly.
merlin_cfg = PIPELINE_CONFIG.merlin
ANALYSIS_TASKS_DIR   = MERCI_DIR / "data" / "configs" / "merlin" / "analysis" / "tasks"
N_OPTIMIZE_ITERATIONS = info.extra.get("n_opt", 10)   # MERlin param, independent of bit count; reused below (snakemake cell)

generated_recipe_path = MERLIN_DIR / "analysis" / f"recipe_{SHORT_NAME}.yaml"
generated_recipe_path.parent.mkdir(parents=True, exist_ok=True)
import yaml as _yaml
with open(generated_recipe_path, "w") as fh:
    _yaml.safe_dump({"n_optimize_iterations": N_OPTIMIZE_ITERATIONS, "tasks": merlin_cfg.enabled_tasks}, fh, sort_keys=False)
print(f"Generated recipe (from pipeline.yaml): {generated_recipe_path}")

analysis_path = MERLIN_DIR / "analysis" / f"merlin_analysis_{SHORT_NAME}.yaml"
build_merlin_analysis_parameters(
    recipe_path            = generated_recipe_path,
    tasks_dir               = ANALYSIS_TASKS_DIR,
    output_path              = analysis_path,
    n_optimize_iterations     = N_OPTIMIZE_ITERATIONS,
    extra_tasks                = [],
    overrides                   = dict(merlin_cfg.overrides),
)
print(f"Saved: {analysis_path}")
display_file(analysis_path)

## Snakemake cluster-resource-allocation + parameters JSON

In [ ]:
cluster_template = MERCI_DIR / "data" / "configs" / "merlin" / "snakemake" / "cluster_resource_allocation_basic.json"

cluster_resource_path = MERLIN_DIR / "snakemake" / f"cluster_resource_allocation_{SHORT_NAME}.json"
create_cluster_resource_allocation(
    template_path=cluster_template, exp_name=SAMPLE_NAME,
    n_optimize_iterations=N_OPTIMIZE_ITERATIONS, output_path=cluster_resource_path,
)
print(f"Saved: {cluster_resource_path}")
display_file(cluster_resource_path)

snakemake_params_path = MERLIN_DIR / "snakemake" / f"parameters_{SHORT_NAME}.json"
create_snakemake_parameters(
    exp_name=SAMPLE_NAME, cluster_config_path=cluster_resource_path,
    output_path=snakemake_params_path,
    job_name_prefix=SHORT_NAME,
)
print(f"Saved: {snakemake_params_path}")
display_file(snakemake_params_path)

## Slurm submit script

Every path in the generated script is written relative to one `$SAMPLE_DIR` bash variable (this experiment's acquisition root — see `resolve_cluster_sample_dir`), instead of five separately-resolved absolute paths. This makes the script identical whether it was generated here (Windows, before transferring `SAMPLE_DIR` to the cluster — `$SAMPLE_DIR` is then *predicted* from the sample name) or regenerated on the cluster itself after the transfer (Linux — `$SAMPLE_DIR` is just the real current path, no guessing).

In [ ]:
data_org_path  = MERLIN_DIR / "dataorganization" / f"data_organization_{MICROSCOPE.upper()}_{SAMPLE_NAME}.csv"
positions_path = POSITIONS_DIR / f"positions_{POSITIONS_TAG}.txt"

# Use the camera-rotation-corrected positions file if notebooks/misc/
# correct_camera_rotation.ipynb has been run for this experiment -- MERlin
# should always decode against the corrected FOV positions when available,
# since the raw grid assumes zero camera-vs-stage rotation.
corrected_positions_path = POSITIONS_DIR / f"positions_{POSITIONS_TAG}_corrected.txt"
if corrected_positions_path.exists():
    positions_path = corrected_positions_path
    print(f"Using camera-rotation-corrected positions file: {positions_path.name} "
          f"(see notebooks/misc/correct_camera_rotation.ipynb)")

for p in (data_org_path, positions_path):
    if not p.exists():
        print(f"WARNING: {p} not found. For a multi-boundary experiment, set the "
              f"correct per-segment file path manually before submitting to the cluster.")

# $SAMPLE_DIR in the generated script: this experiment's acquisition root as it
# will be addressed from the Linux cluster. Predicted from the TRUE sample_name +
# imaging_dir (experiment_info.yaml, notebook 06) when running on Windows
# (before transfer); the real current path when already running on the
# cluster (Linux) -- see resolve_cluster_sample_dir.
sample_dir_cluster = resolve_cluster_sample_dir(SAMPLE_DIR, SAMPLE_NAME, info.extra.get("imaging_dir", ""))

def _rel(p):
    """POSIX path relative to SAMPLE_DIR, for use as "$SAMPLE_DIR/<...>" in the script."""
    return Path(p).relative_to(SAMPLE_DIR).as_posix()

submit_path = MERLIN_DIR / "slurm" / "submit" / f"merlin_slurm_{SHORT_NAME}.sh"
create_slurm_submit_script(
    label                   = SHORT_NAME,
    sample_dir              = sample_dir_cluster,
    parameters_file         = _rel(snakemake_params_path),
    analysis_file           = _rel(analysis_path),
    data_organization_file  = _rel(data_org_path),
    positions_file          = _rel(positions_path),
    codebook_file           = _rel(codebook_path),
    microscope_file         = _rel(microscope_path),
    data_home               = info.data_home,
    folder_name             = info.folder_name,
    output_path             = submit_path,
    analysis_name           = "output",
)
print(f"$SAMPLE_DIR (cluster) : {sample_dir_cluster}")
print(f"Saved: {submit_path}\n")
display_file(submit_path)